## Triangle Color Rasterisation

#### Objective:
- $\text{Show how to define explicitely a 2D linear function interpolating the values inside the triangle based on the values of the vertices}$
- $\text{Review the more efficient scan-line triangle rasterisation algorithm and explain how color values, are calculated in this algorithm}$

- $\text{The downside of calculating the barycentric coordinate was that for each point (x,y) inside the triangle, there's a need to calculate the coordinates all over again}$
- $\text{This includes calculating the area of three triangles which is equivelant to calculating 3 determinants}$
- $\text{It's possible to show that interpolating using barycentric coordinates, actually defines a 2D linear function over the triangle interpolating the values of the vertices.}$

### Optimaisation
- $\text{We need to find a linear function interpolating the three scalar values defined based on the vertices}$
- $\text{These values can be thought of as the z-value of the 2D vertices, this means we want a linear function that interpolates three 3D points}$

<p align="center">
 <img src="image_U11/Screenshot 2025-06-20 at 22.20.02.png" height="400" width="500"/>
   <img src="image_U11/Screenshot 2025-06-20 at 22.20.47.png" height="400" width="500"/>
<p/>

$\text{Formally: we have 3 vertices } (x_1, y_1), (x_2, y_2), (x_3, y_3) \text{ that have three scalar values } f_1, f_2, f_3 \text{respectively. We define a plane } f(x,y) = Ax + By + c \text{such that:}$
$\text{The plane passes through the 3D points } (x_1, y_1, f1), (x_2, y_2, f2), (x_3, y_3, f3)$
$\text{ We have 3 unknowns (A, B, C) and 3 equations define as follows:}$
$$ f(x_1, y_1) = Ax_1 + By_1 + C = f_1 $$
$$ f(x_2, y_2) = Ax_2 + By_2 + C = f_2 $$
$$ f(x_1, y_1) = Ax_3 + By_3 + C = f_3 $$

$$
\begin{bmatrix}
x_1 & y_1 & 1 \\
x_2 & y_2 & 1 \\
x_3 & y_3 & 1
\end{bmatrix}
\begin{bmatrix}
A \\
B \\
C
\end{bmatrix}
=
\begin{bmatrix}
f_1 \\
f_2 \\
f_3
\end{bmatrix}
$$

$$
\begin{bmatrix}
A \\
B \\
C
\end{bmatrix}
=
\begin{bmatrix}
x_1 & y_1 & 1 \\
x_2 & y_2 & 1 \\
x_3 & y_3 & 1
\end{bmatrix}^{-1}
\begin{bmatrix}
f_1 \\
f_2 \\
f_3
\end{bmatrix}
$$

$\text{Once we have solved for A, B and C we can use the function f(x,y) to find any point inside the triangle}$
$\text{To interpolate color we need to find A, B and C values defining the equation for each of the three component red, green and blue}$
$$
\begin{bmatrix}
A_{red} \\
B_{red} \\
C_{red}
\end{bmatrix}
=
\begin{bmatrix}
x_1 & y_1 & 1 \\
x_2 & y_2 & 1 \\
x_3 & y_3 & 1
\end{bmatrix}^{-1}
\begin{bmatrix}
R_1 \\
R_2 \\
R_3
\end{bmatrix}
$$

$$
\begin{bmatrix}
A_{green} \\
B_{green} \\
C_{green}
\end{bmatrix}
=
\begin{bmatrix}
x_1 & y_1 & 1 \\
x_2 & y_2 & 1 \\
x_3 & y_3 & 1
\end{bmatrix}^{-1}
\begin{bmatrix}
G_1 \\
G_2 \\
G_3
\end{bmatrix}
$$

$$
\begin{bmatrix}
A_{blue} \\
B_{blue} \\
C_{blue}
\end{bmatrix}
=
\begin{bmatrix}
x_1 & y_1 & 1 \\
x_2 & y_2 & 1 \\
x_3 & y_3 & 1
\end{bmatrix}^{-1}
\begin{bmatrix}
B_1 \\
B_2 \\
B_3
\end{bmatrix}
$$

### Incoorperating Color interpolation in Scan-Line Algorithm

- $\text{In the previous algorithms we still need to use the barycentric method to determine whether a point was inside the triangle, even with the above method, this optimised algorithm will be slow}$
- $\text{If we use the scan-line algorithm we don't wish to use the barycentric (to calculate the color) since it'll slow down to computation time drastically}$
- $\text{in the scane line algorithm: }$


<p align="center">
 <img src="image_U11/Screenshot 2025-06-20 at 22.38.14.png" height="400" width="500"/>
<img src="image_U11/Screenshot 2025-06-20 at 22.39.49.png" height="400" width="500"/>
<p/>

1. $\text{Rasterises the left and right edges, where the requirement was that the color interpolation on the edges would be linear between the color values, of the vertices definning the edges}$
    - $\text{in the scanline algorithm, while we rasterise the righ and left edge pixels, we can calulate their colors as a linear combination fo teh vertice' colors}$
    - $\text{The algorithm draws the span of pixels in each row we want to avoid calculating the expensive barycentric coordinates}$
    - $\text{Since we already know the left and right color pixels of the scan edge, we can use linear interpolation between these colors to define the color fo the pixel between them}$

| Feature                         | Bounding Box (Barycentric)                              | Scan-Line Algorithm                               |
|----------------------------------|----------------------------------------------------------|----------------------------------------------------|
| **Approach**                    | Iterate over a 2D bounding box around triangle           | Process horizontal lines intersecting the triangle |
| **Key Equation**                | Barycentric coords:  $$ P = \lambda_1 A + \lambda_2 B + \lambda_3 C $$  | Edge intersections per scanline                    |
| **Point-In-Triangle Test**      | $$ \lambda_1, \lambda_2, \lambda_3 \geq 0,\ \lambda_1 + \lambda_2 + \lambda_3 = 1 $$ | Use edge functions or parity test                  |
| **Setup Complexity**           | Simple (bounding box + barycentric weights)             | Higher (requires edge sorting and scanline tracking) |
| **Fill Accuracy**              | Uniform subpixel coverage possible                       | May suffer from aliasing if not supersampled       |
| **Parallelization**            | Easy — each pixel checked independently                  | Harder — scanline logic is more sequential         |
| **Efficiency**                 | Efficient in GPUs and tile-based rasterizers             | Efficient in software pipelines                    |

